# Ankur Task 06C evaluation

This provider-free notebook keeps the frozen Task 06 historical result separate from Task 06C. It reads only committed public-safe normalized exports and never requires an API key. The single Task 06C live run failed two measured gates before human review; human-dependent metrics remain pending.

In [1]:
import json
from pathlib import Path

ROOT = Path('evaluation/task06c')
manifest = json.loads((ROOT / 'corpus/public/manifest.json').read_text(encoding='utf-8'))
pending_metrics = json.loads((ROOT / 'exports/task06c-metrics.pending.json').read_text(encoding='utf-8'))
pending_gates = json.loads((ROOT / 'exports/task06c-gate-status.pending.json').read_text(encoding='utf-8'))
metrics = json.loads((ROOT / 'exports/task06c-metrics.live-run.json').read_text(encoding='utf-8'))
gates = json.loads((ROOT / 'exports/task06c-gate-status.live-run.json').read_text(encoding='utf-8'))
operations = json.loads((ROOT / 'records/public/provider-operations.json').read_text(encoding='utf-8'))
questions = json.loads((ROOT / 'records/public/question-records.json').read_text(encoding='utf-8'))
written = json.loads((ROOT / 'records/public/written-grading-records.json').read_text(encoding='utf-8'))
baseline = json.loads((ROOT / 'records/public/baseline-records.json').read_text(encoding='utf-8'))
historical = json.loads(Path(metrics['historicalTask06MetricsPath']).read_text(encoding='utf-8'))
print(f"Task 06C corpus: {manifest['frozenMaterialCount']} frozen + {manifest['holdoutMaterialCount']} holdout")
print(f"Historical Task 06 quality gate: {historical['gate']['productQualityGate']}")
print(f"Task 06C live-run gate: {gates['overallStatus']}")

Task 06C corpus: 6 frozen + 3 holdout
Historical Task 06 quality gate: failed
Task 06C live-run gate: failed


In [2]:
assert manifest['frozenMaterialCount'] == 6
assert manifest['holdoutMaterialCount'] >= 3
assert len(manifest['materials']) >= 9
assert pending_metrics['providerAttempts'] == 0
assert pending_gates['overallStatus'] == 'pending'
assert metrics['materials'] == {'frozen': 6, 'holdout': 3, 'total': 9}
logical_operations = len(operations)
provider_attempts = logical_operations + sum(1 for operation in operations if operation['repairAttempted'])
final_valid = sum(1 for operation in operations if operation['finalStatus'] == 'valid')
baseline_questions = sum(record['parsedQuestionCount'] for record in baseline)
assert (logical_operations, provider_attempts, final_valid) == (45, 72, 33)
assert metrics['logicalOperations'] == logical_operations
assert metrics['providerAttempts'] == provider_attempts
assert metrics['finalLogicalArtifactValidity']['numerator'] == final_valid
assert metrics['finalLogicalArtifactValidity']['denominator'] == logical_operations
assert (len(questions), baseline_questions, len(written)) == (18, 42, 7)
assert metrics['eligibleWrittenCases'] == len(written)
print('Live-run denominators reconciled: 45 logical operations, 72 provider attempts, 33 final-valid operations.')
print('Artifacts reconciled: 18 Ankur questions, 42 baseline questions, 7 written cases.')

Live-run denominators reconciled: 45 logical operations, 72 provider attempts, 33 final-valid operations.
Artifacts reconciled: 18 Ankur questions, 42 baseline questions, 7 written cases.


In [3]:
pending = [gate for gate in gates['gates'] if gate['status'] == 'pending']
failed = [gate for gate in gates['gates'] if gate['status'] == 'failed']
assert gates['overallStatus'] == 'failed'
assert gates['task07Authorized'] is False
assert [gate['gate'] for gate in failed] == ['eligible_written_cases', 'final_logical_artifact_validity']
assert metrics['humanReviewStatus'] == 'pending'
print('Failed measured gates: ' + ', '.join(gate['gate'] for gate in failed))
print(f"Pending human-dependent gates: {len(pending)}")
print('TASK 07 remains blocked.')

Failed measured gates: eligible_written_cases, final_logical_artifact_validity
Pending human-dependent gates: 7
TASK 07 remains blocked.
